# Notebook 14: Type Casting in C++

C-style casts exist in C++ but are dangerous. C++ provides **named casts** that are safer, more explicit, and grep-able.

A named cast tells both the compiler and the human reader exactly what kind of conversion is happening — and why it is (or is not) safe.

## C-Style Casts (Review)

You know these from C:

```c
int x = (int)3.14;
double d = (double)x;
```

They work in C++ — but they are **discouraged** for several reasons:

1. **Too powerful** — a single C-style cast can do many different things depending on context: numeric conversion, const removal, pointer reinterpretation. It is easy to do the wrong one accidentally.
2. **Hard to search** — `grep '(int)'` in a large codebase is noisy. `grep 'static_cast'` finds only your explicit casts.
3. **No safety checks** — the compiler accepts almost any C-style cast without questioning whether it makes sense.

In [ ]:
#include <iostream>

double pi = 3.14159;
int truncated = (int)pi;  // C-style: works, but discouraged
std::cout << "C-style (int)3.14159 = " << truncated << std::endl;

// C-style can silently remove const:
const int LIMIT = 100;
int *p = (int *)&LIMIT;   // C-style: compiles, removes const — dangerous!
std::cout << "C-style removed const (dangerous)" << std::endl;
(void)p;

## The Four Named Casts — Overview

| Cast | When to use | Safety |
|---|---|---|
| `static_cast<T>(x)` | Compile-time checked conversions: numeric, up/downcast | Safe for well-known conversions |
| `dynamic_cast<T>(x)` | Runtime-checked cast for polymorphic types | Safe (returns `nullptr` on failure) — **excluded from this course** |
| `const_cast<T>(x)` | Add or remove `const` qualifier | Rarely needed — **just know it exists** |
| `reinterpret_cast<T>(x)` | Raw bit reinterpretation between unrelated types | Dangerous — low-level use only |

This notebook focuses on **`static_cast`**, which covers the vast majority of legitimate casting needs.

## `static_cast`: Basic Numeric Conversions

`static_cast` is the correct tool for converting between numeric types. The compiler checks the conversion at compile time and rejects conversions that make no sense.

In [ ]:
#include <iostream>

// Double to int: truncation (not rounding)
double d = 3.99;
int i = static_cast<int>(d);
std::cout << "static_cast<int>(3.99) = " << i << std::endl;  // 3, not 4

// The integer division trap
int a = 7;
int b = 2;

double bad = a / b;  // Integer division: 3, then promoted to 3.0
std::cout << "a / b (integer division) = " << bad << std::endl;

double good = static_cast<double>(a) / b;  // Cast a to double first, then divide
std::cout << "static_cast<double>(a) / b = " << good << std::endl;

**Exercise 1:** Fix this expression so it gives the correct floating-point result `3.5` instead of `3.0`:

```cpp
int a = 7;
int b = 2;
double result = a / b;  // currently gives 3.0
```

Use `static_cast` to fix it. Print both the broken and fixed versions.

In [ ]:
// Your code here

## `static_cast` with `char`

`char` is an integer type in C and C++. You can convert between `int` and `char` explicitly to work with ASCII values.

In [ ]:
#include <iostream>

// int to char
char letter = static_cast<char>(65);
std::cout << "static_cast<char>(65) = '" << letter << "'" << std::endl;  // 'A'

// char to int
char ch = 'A';
int code = static_cast<int>(ch);
std::cout << "static_cast<int>('A') = " << code << std::endl;  // 65

// Print ASCII range for printable characters
std::cout << "Printable ASCII: ";
for (int n = 32; n < 127; n++) {
    std::cout << static_cast<char>(n);
}
std::cout << std::endl;

## `static_cast` vs C-Style Cast

Comparing the two side by side:

In [ ]:
#include <iostream>

double value = 9.7;

// C-style: works but opaque
int cStyle = (int)value;

// static_cast: explicit about intent
int cppStyle = static_cast<int>(value);

std::cout << "C-style:     " << cStyle  << std::endl;
std::cout << "static_cast: " << cppStyle << std::endl;

// static_cast catches impossible conversions at COMPILE TIME:
// struct Foo {};
// struct Bar {};
// Foo f;
// Bar b = static_cast<Bar>(f);  // COMPILE ERROR — no relationship between Foo and Bar
// (int)f;                        // Also error in this case, but not always

std::cout << "static_cast is searchable: grep 'static_cast' finds all casts" << std::endl;

## Upcasting (Derived to Base)

Converting a derived pointer/reference to a base pointer/reference is called **upcasting** — moving up the inheritance hierarchy.

Upcasting is always safe because a `Derived` object **is a** `Base` object. C++ performs this implicitly, but you can make it explicit with `static_cast`.

In [ ]:
#include <iostream>
#include <string>

class Animal {
public:
    std::string name;
    Animal(const std::string &n) : name(n) {}
    virtual void speak() const { std::cout << name << " makes a sound" << std::endl; }
    virtual ~Animal() {}
};

class Dog : public Animal {
public:
    Dog(const std::string &n) : Animal(n) {}
    void speak() const { std::cout << name << " says: Woof!" << std::endl; }
    void fetch() const { std::cout << name << " fetches the ball" << std::endl; }
};

Dog *dog = new Dog("Rex");

// Implicit upcast — happens automatically
Animal *animalImplicit = dog;

// Explicit upcast with static_cast — same result, clearer intent
Animal *animalExplicit = static_cast<Animal *>(dog);

animalImplicit->speak();   // Rex says: Woof! (virtual dispatch)
animalExplicit->speak();   // Rex says: Woof!

delete dog;

## Downcasting (Base to Derived)

Converting a base pointer to a derived pointer is called **downcasting** — moving down the hierarchy.

This is **only safe if you know** the object actually is of the derived type. `static_cast` does not check at runtime — if you are wrong, you get **undefined behaviour**.

> **Warning:** Only downcast when you are certain of the actual type (e.g. you stored it yourself). If you need runtime safety, that is what `dynamic_cast` is for (excluded from this course).

In [ ]:
#include <iostream>

// We know this Animal* actually points to a Dog
Animal *base = new Dog("Buddy");

// Safe downcast: we KNOW it is a Dog
Dog *derived = static_cast<Dog *>(base);
derived->fetch();  // Access Dog-specific method
derived->speak();

// DANGER — do not do this:
// class Cat : public Animal { ... };
// Cat *wrongCast = static_cast<Cat *>(base);  // Compiles but UNDEFINED BEHAVIOUR
// wrongCast->purr();  // Anything could happen

delete base;

**Exercise 2:** Create a `Shape` base class (with a virtual destructor) and a `Rectangle` derived class with `width` and `height` and a method `getWidth()`. Create a `Shape *` that actually holds a `Rectangle`. Use `static_cast` to downcast it to `Rectangle *` and call `getWidth()`. In a comment, explain what would happen if you tried to downcast it to a hypothetical `Circle *` instead.

In [ ]:
// Your code here

## Why Not `dynamic_cast`?

`dynamic_cast` performs a runtime type check and returns `nullptr` if the cast is incorrect:

```cpp
Dog *safe = dynamic_cast<Dog *>(animalPtr);
if (safe != NULL) {
    safe->fetch();
}
```

We exclude it from this course because:
- It requires **RTTI** (Run-Time Type Information) — extra overhead kept by the compiler
- It requires the hierarchy to have at least one `virtual` method
- In well-designed code with proper polymorphism, downcasting is often a sign of a **design problem** — you should not need to ask "what type is this?". If you find yourself needing `dynamic_cast` frequently, consider redesigning with more virtual methods.

## `reinterpret_cast`

`reinterpret_cast` treats the raw bits of one type as another type — no conversion, just reinterpretation.

Use cases: low-level memory inspection, binary serialisation, hardware register access.

> This is genuinely dangerous. Only use it if you know exactly what the memory layout is.

In [ ]:
#include <iostream>

// Safe use: inspect the raw bytes of an int
int value = 0x41424344;  // ASCII 'A','B','C','D' on little-endian

unsigned char *bytes = reinterpret_cast<unsigned char *>(&value);

std::cout << "Bytes of int 0x41424344 (hex): ";
for (int n = 0; n < static_cast<int>(sizeof(int)); n++) {
    std::cout << std::hex << static_cast<int>(bytes[n]) << " ";
}
std::cout << std::dec << std::endl;

// On little-endian: prints 44 43 42 41 (reversed)
// On big-endian: prints 41 42 43 44

## Implicit Conversions

C++ performs some conversions **automatically** — no cast required:

- `int` → `double` (widening: safe, no data loss)
- `Derived *` → `Base *` (upcasting: always safe)
- Non-`const` → `const` (always safe: adding restrictions)
- `float` → `double` (widening)

These are all safe and intentional. Be aware of them — sometimes an implicit conversion produces a surprising result.

In [ ]:
#include <iostream>

int i2 = 42;
double d2 = i2;  // Implicit: int → double, no cast needed
std::cout << "Implicit int->double: " << d2 << std::endl;

// Non-const to const: always fine
int x = 10;
const int &ref = x;  // Implicit: adding const
std::cout << "const ref: " << ref << std::endl;

// Watch out: narrowing is also implicit in C++98 (just a warning, not an error)
double big = 1234567890.5;
int narrow = big;  // Implicit narrowing — data loss!
std::cout << "Implicit narrowing (data loss!): " << narrow << std::endl;
// In C++11, narrowing in {} initialisation is an ERROR (see Modern section)

## Final Exercise

1. Write a function `int safeDivide(int numerator, int denominator)` that returns the integer result of division. The function should just return `numerator / denominator` — no casting needed since we want integer result. Add a comment explaining why no cast is needed here.

2. Write a function `double precisionDivide(int a, int b)` that returns the **exact** floating-point result of dividing `a` by `b`. Use `static_cast` to ensure floating-point division.

3. Test both functions: `safeDivide(7, 2)` should give `3`, `precisionDivide(7, 2)` should give `3.5`.

In [ ]:
// Your code here

## Modern C++ (C++11 and Beyond)

### `explicit` Constructors
A constructor taking one argument can be called implicitly in C++98. The `explicit` keyword prevents this, avoiding unexpected implicit conversions.

### `explicit` Conversion Operators
C++11 also allows `explicit` on conversion operators: `explicit operator bool()` — used e.g. for `if (stream)` without allowing arithmetic on streams.

### Narrowing Conversions in Uniform Initialisation
In C++98, `int x = 3.14;` compiles with just a warning. In C++11, `int x{3.14};` is a **compile error** — narrowing conversions are forbidden inside `{}`.

In [ ]:
#include <iostream>

class Meters {
public:
    double value;

    explicit Meters(double v) : value(v) {}  // 'explicit': must construct deliberately

    explicit operator double() const { return value; }  // explicit conversion op
};

Meters m(5.0);              // OK: explicit construction
// Meters m2 = 5.0;         // ERROR: implicit conversion prevented by 'explicit'

double raw = static_cast<double>(m);  // OK: explicit cast
// double raw2 = m;                   // ERROR: implicit conversion operator is explicit

std::cout << "Meters value: " << m.value << std::endl;
std::cout << "Cast to double: " << raw << std::endl;

// Narrowing in uniform initialisation (C++11)
double pi = 3.14159;
// int bad{pi};   // COMPILE ERROR in C++11: narrowing conversion not allowed in {}
int ok = pi;     // C++98-style: compiles (with warning)
std::cout << "C++98 narrowing: " << ok << std::endl;